# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

**Design decisions (from the data dictionary and a look at the structure):**
- There is no date/time column, so there is no "later rows" to forecast; the relevant generalisation is to **new students**.
- `student_key` repeats (about 7 rows per student), so splits are by **whole student** (checklist 1, 2).
- `student_key` is an identifier and is dropped from the features.
- Prediction time: the end of the session, just before the quiz. `session_minutes` and `hints_used` are recorded in the session, so they are available then.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, brier_score_loss
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.student_key.nunique(), 'students'); print(df.passed_quiz.mean())
print(df.groupby('student_key')[['school_code','enrol_year']].nunique().max())

## 1. Test set: held-out whole students (set aside before any modelling)

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr_idx,te_idx=next(gss.split(df,groups=df.student_key))
train,test=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key)&set(test.student_key)
print(len(train),len(test),train.passed_quiz.mean())

## 2. Candidates (named before any scoring)
Feature sets: **A** = `prior_score, session_minutes, hints_used`; **B** = A + `enrol_year`, `school_code` (one-hot).
`student_key` is never a feature.

| name | model |
|---|---|
| baseline | predicts training prevalence (`DummyClassifier(prior)`) |
| logreg_A_C0.1, logreg_A_C1 | logistic regression on A |
| logreg_B_C0.1, logreg_B_C1 | logistic regression on B |
| hgb_A_d2, hgb_A_d3 | gradient boosting on A (max_depth 2/3) |
| hgb_B_d2, hgb_B_d3 | gradient boosting on B |

Selection metric: cross-validated ROC AUC (pooled out-of-fold). The decision threshold is then chosen on out-of-fold predictions by balanced accuracy.

In [ ]:
NUM=['prior_score','session_minutes','hints_used']; EXTRA=['enrol_year','school_code']
def prep(cols, scale):
    num=[c for c in cols if c!='school_code']
    t=[('n',StandardScaler() if scale else 'passthrough',num)]
    if 'school_code' in cols: t.append(('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['school_code']))
    return ColumnTransformer(t)
def mk(kind,fs,p):
    cols=NUM if fs=='A' else NUM+EXTRA
    if kind=='logreg': return cols,make_pipeline(prep(cols,True),LogisticRegression(C=p,max_iter=2000))
    return cols,make_pipeline(prep(cols,False),HistGradientBoostingClassifier(max_depth=p,max_iter=150,learning_rate=0.05,random_state=SEED))
cands={'baseline':(NUM,make_pipeline(DummyClassifier(strategy='prior')))}
for fs in 'AB':
    for C in (0.1,1): cands[f'logreg_{fs}_C{C}']=mk('logreg',fs,C)
    for d in (2,3): cands[f'hgb_{fs}_d{d}']=mk('hgb',fs,d)
print(list(cands))

## 3. Grouped 5-fold CV on training rows only (preprocessing lives inside each pipeline, refit per fold)

In [ ]:
from sklearn.base import clone
gkf=GroupKFold(n_splits=5)
folds=list(gkf.split(train,groups=train.student_key))
y=train.passed_quiz.values
oof={}; rows=[]
for name,(cols,pipe) in cands.items():
    p=np.zeros(len(train)); aucs=[]
    for a,b in folds:
        m=clone(pipe).fit(train.iloc[a][cols],y[a]); p[b]=m.predict_proba(train.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(y[b],p[b]))
    oof[name]=p; rows.append((name,np.mean(aucs),np.std(aucs),brier_score_loss(y,p)))
cv=pd.DataFrame(rows,columns=['model','cv_auc','auc_sd_over_folds','brier']).sort_values('cv_auc',ascending=False)
cv

## 4. Choice and threshold (CV only)

In [ ]:
best=cv.iloc[0].model
ths=np.linspace(0.05,0.95,91)
bal=[balanced_accuracy_score(y,oof[best]>=t) for t in ths]
thr=float(ths[int(np.argmax(bal))])
print('chosen model:',best,'| threshold:',thr)

**Recorded choice (written before the test set is scored):** the model and threshold printed above — the highest cross-validated AUC, with the threshold maximising balanced accuracy on out-of-fold predictions. Nothing below changes them.

## 5. Refit on all training rows; score the test set once

In [ ]:
cols,pipe=cands[best]
final=clone(pipe).fit(train[cols],y)
pt=final.predict_proba(test[cols])[:,1]; yt=test.passed_quiz.values
base=clone(cands['baseline'][1]).fit(train[NUM],y).predict_proba(test[NUM])[:,1]
pred=pt>=thr
res=pd.Series({'test_auc':roc_auc_score(yt,pt),'baseline_auc':roc_auc_score(yt,base),
 'balanced_acc':balanced_accuracy_score(yt,pred),'accuracy':accuracy_score(yt,pred),
 'majority_class_accuracy':1-yt.mean(),'brier':brier_score_loss(yt,pt),'brier_baseline':brier_score_loss(yt,base)})
rng=np.random.default_rng(SEED); st=test.student_key.values; ids=np.unique(st)
ix={s:np.where(st==s)[0] for s in ids}; bs=[]
for _ in range(500):
    j=np.concatenate([ix[s] for s in rng.choice(ids,len(ids))]); bs.append(roc_auc_score(yt[j],pt[j]))
print(res.round(4)); print('AUC 95% CI (student bootstrap):',np.percentile(bs,[2.5,97.5]).round(3))

## 6. Interpretation
The headline is the single test-set AUC above, compared with the baseline AUC (0.5 by construction). Caveats:
- Split is by student; with no timestamp we cannot say how the model does on future sessions.
- `prior_score` is described as "as stored on the session record" and varies within a student, so it may not be a fixed placement score; confirm when it is written before relying on it in deployment.
- `session_minutes` may include quiz time; if the model must predict before the quiz starts, confirm this column is available then.
- The test set was scored once; no changes were made after seeing it.